# Hello KCT

**What it does**
Sends one question, "What is 5+2?", to the AI provider through the app's `AskService` and prints the answer, the model that answered, the tokens used and how long it took.

**Why**
It is the smallest check that everything is wired up: the project's code can be imported, the settings are read, and the AI provider answers. Run it first on a new machine, after installing packages, or after changing `KCT_AI_PROVIDER`, before you spend time and money on a real interpretation run. The API does the same through `POST /api/v1/ask`, so a working notebook means the API's code works too.

**How to use**
1. Open this notebook from the project's `notebooks/` folder.
2. Choose the provider: in the project's `.env` file, or by uncommenting the `os.environ[...]` lines in the first code cell (those win over `.env`). With neither it is `fake`, which needs no cloud access and always answers `5 + 2 = 7.` (from `tests/fixtures/fake_ai/ask.md`).
3. Run all cells (Run > Run All Cells).
4. Change the question in the second code cell if you like.
5. After changing a setting, restart the kernel (Kernel > Restart Kernel): settings are read once per kernel.

**Needs**: `pydantic` 2.x, and `google-genai` for Gemini. Terminal equivalent: `./kct.sh ask`.

## Setup

Finds the project folder so `services` can be imported, picks the AI provider, and builds the app's services with `boot()`. Change the provider here.

In [ ]:
import os, sys
from pathlib import Path

# The project folder: the parent of notebooks/. Change it if you moved the notebook.
PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

# Settings come from the project's .env and from lines like these, which win over .env:
# os.environ["KCT_AI_PROVIDER"] = "gemini"
# os.environ["KCT_PROJECT_ID"] = "<project>"
# They are read once per kernel: restart the kernel after changing them.

from services.app import boot
from services.config import load_env_file

load_env_file()
os.environ.setdefault("KCT_AI_PROVIDER", "fake")  # nothing chosen anywhere: no cloud
print("AI provider:", os.environ["KCT_AI_PROVIDER"])

kct = boot()

## Ask the question

Sends the question and prints the answer. Change the text to ask something else.

In [ ]:
try:
    result = kct.ask.ask("What is 5+2?")
except Exception as exc:
    # Usually a missing setting (KCT_PROJECT_ID) or a missing Google login.
    print("❌", type(exc).__name__, exc)
else:
    print(result.answer)


## Full result: provider, model, tokens, time

In [ ]:
if "result" in globals():
    print(result.model_dump_json(indent=2))
